# Context Windows, API Costs, and Token Limits in LLMs

This notebook explains how context windows, token limits, and API costs affect LLM application design.

In [1]:
%pip install openai python-dotenv tiktoken

  Using cached tiktoken-0.12.0-cp314-cp314-win_amd64.whl.metadata (6.9 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached regex-2026.4.4-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ---------------------------------------- 0.0/1.2 MB 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"
encoding = tiktoken.encoding_for_model(MODEL)

## What is a Context Window?

A context window is the maximum amount of text a model can consider in one request.

It includes:
- system message
- user message
- assistant message history
- retrieved documents
- tool outputs
- the model's final answer

If the input is too large, the model may reject it or you must shorten the prompt.

In [3]:
# Count tokens in text
def count_tokens(text: str) -> int:
    return len(encoding.encode(text))

examples = [
    "Hello world.",
    "Explain retrieval augmented generation in simple terms.",
    "Prior authorization requires approval before certain medications are covered."
]

for text in examples:
    print(f"Text: {text}")
    print(f"Token count: {count_tokens(text)}")
    print("-" * 60)

Text: Hello world.
Token count: 3
------------------------------------------------------------
Text: Explain retrieval augmented generation in simple terms.
Token count: 8
------------------------------------------------------------
Text: Prior authorization requires approval before certain medications are covered.
Token count: 10
------------------------------------------------------------


In [4]:
# Count approximate tokens in chat messages
def count_message_tokens(messages: list[dict]) -> int:
    text = ""
    for message in messages:
        text += f"{message['role']}: {message['content']}\n"
    return count_tokens(text)

messages = [
    {"role": "system", "content": "You are a helpful AI assistant"},
    {"role": "user", "content": "Explain what a context window is."}
]

print("Approx message tokens:", count_message_tokens(messages))

Approx message tokens: 18


In [5]:
# Simulate growing conversation history
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."}
]

for i in range(1, 11):
    messages.append({"role": "user", "content": f"This is user message number {i}. Please remember it."})
    messages.append({"role": "assistant", "content": f"I will remember message number {i}."})

    print(f"After turn {i}: {count_message_tokens(messages)} approximate tokens")

After turn 1: 33 approximate tokens
After turn 2: 57 approximate tokens
After turn 3: 81 approximate tokens
After turn 4: 105 approximate tokens
After turn 5: 129 approximate tokens
After turn 6: 153 approximate tokens
After turn 7: 177 approximate tokens
After turn 8: 201 approximate tokens
After turn 9: 225 approximate tokens
After turn 10: 249 approximate tokens


## Why Long Chat Memory Becomes Expensive

Every time we send a request, previous messages must be sent again if we want the model to use them.

That means conversation memory increases:
- prompt size
- cost
- latency
- risk of hitting context limits

In [6]:
# Example prices only. Replace with current provider pricing when needed.
INPUT_COST_PER_1M_TOKENS = 0.15
OUTPUT_COST_PER_1M_TOKENS = 0.60

def estimate_cost(input_tokens: int, output_tokens: int) -> float:
    input_cost = (input_tokens  / 1_000_000) * INPUT_COST_PER_1M_TOKENS
    output_cost = (output_tokens / 1_000_000) * OUTPUT_COST_PER_1M_TOKENS
    return input_cost + output_cost

input_tokens = 2_000
output_tokens = 500

cost = estimate_cost(input_tokens, output_tokens)
print(f"Estimated Cost: ${cost:.6f}")

Estimated Cost: $0.000600


In [7]:
# Real API call with token usage
messages = [
    {"role": "system", "content": "You are a concise AI tutor."},
    {"role": "user", "content": "Explain API costs and token limits in simple terms."}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.3,
)

reply = response.choices[0].message.content
print(reply)

print("\nUsage:")
print(response.usage)

Sure! 

**API Costs**: When you use an API (Application Programming Interface), you're often charged based on how much you use it. This can include costs per request (each time you ask for data), costs for the amount of data transferred, or subscription fees for access. Think of it like paying for electricity: you pay for how much you use.

**Token Limits**: In the context of APIs, especially those related to language models, a "token" is a piece of text, which can be as short as a character or as long as a word. Token limits refer to the maximum number of tokens you can send to or receive from the API in a single request. For example, if an API has a limit of 1,000 tokens, you can only send or receive that many tokens at once. This is like a size limit on a message you can send.

In summary, API costs are about how much you pay based on usage, and token limits are about how much data you can send or receive at one time.

Usage:
CompletionUsage(completion_tokens=212, prompt_tokens=28, 

In [8]:
# Estimate cost from real usage
prompt_tokens = response.usage.prompt_tokens
completion_tokens = response.usage.completion_tokens

estimated_cost = estimate_cost(prompt_tokens, completion_tokens)

print("Prompt tokens:", prompt_tokens)
print("Completion tokens:", completion_tokens)
print("Total tokens:", response.usage.total_tokens)
print(f"Estimated Cost: ${estimated_cost:.8f}")

Prompt tokens: 28
Completion tokens: 212
Total tokens: 240
Estimated Cost: $0.00013140


## Output Token Limits

You can limit how long the model's answer can be.

This helps control:
- cost
- latency
- verbosity

In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Explain transformers in detail."}
    ],
    temperature=0.3,
    max_tokens=100,
)

print(response.choices[0].message.content)
print("\nUsage:", response.usage)

Transformers are a type of neural network architecture that has revolutionized natural language processing (NLP) and other fields since their introduction in the paper "Attention is All You Need" by Vaswani et al. in 2017. Here’s a detailed explanation of their components and functioning:

### Key Components

1. **Self-Attention Mechanism**:
   - The core innovation of transformers is the self-attention mechanism, which allows the model to weigh the importance of different words in a

Usage: CompletionUsage(completion_tokens=100, prompt_tokens=22, total_tokens=122, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


## How AI Engineers Manage Context Windows

Common strategies:

1. Keep only recent conversation history
2. Summarize older messages
3. Use RAG instead of sending entire documents
4. Chunk long documents
5. Retrieve only relevant chunks
6. Limit output tokens
7. Track token usage in logs

In [10]:
# Simple memory trimming example
def trim_messages(messages: list[dict], max_approx_tokens: int) -> list[dict]:
    system_message = messages[0]
    remaining = messages[1:]

    while count_message_tokens([system_message] + remaining) > max_approx_tokens:
        if remaining:
            remaining.pop(0)
        else:
            break

    return [system_message] + remaining


long_messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

for i in range(1, 20):
    long_messages.append({"role": "user", "content": f"Message {i}: This is some conversation history."})
    long_messages.append({"role": "assistant", "content": f"Response {i}: Acknowledged."})

print("Before trimming:", count_message_tokens(long_messages))

trimmed = trim_messages(long_messages, max_approx_tokens=150)

print("After trimming:", count_message_tokens(trimmed))
print("Messages kept:", len(trimmed))

Before trimming: 426
After trimming: 150
Messages kept: 14


## RAG connection

RAG helps manage context 

Instead of sending an entire document to the model, we:

1. Split the document into chunks
2. Embed each chunk
3. Retrieve only the most relevant chunks
4. Send those chunks as text

This keeps prompts smaller, cheaper, and more focused

## Key Takeaways

- A context window is the text limit the model can process in one request.
- Both input and output tokens count toward API usage.
- Longer prompts increase cost and latency.
- Chat memory is expensive because previous messages must be resent.
- `max_tokens` helps control output length.
- RAG is a practical strategy for managing context windows.